# SENDA Copilot · ajuste fino con LoRA (gratis en Google Colab) · ronda 2

Entrena **Qwen3 1.7B** para que sea el copiloto de SENDA: elegir la herramienta correcta, entender
las órdenes (tiempo, tormenta, objetivos, perfil), explicar cómo se usa el sistema y responder
solo con las cifras del planificador.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

Pasos: instalar → cargar datos → cargar modelo → LoRA → entrenar → probar → exportar a GGUF.
Tiempo aproximado en una T4: 1–2 h de entrenamiento + 15 min de exportación.

Los datos los genera `backend/training/generate_dataset.py` con el planificador real
(`train.jsonl` y `val.jsonl`). El examen `backend/eval/run_eval.py` mide antes y después.

## 1. Instalar Unsloth

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo

## 2. Cargar los datos

Opción A: sube `train.jsonl` y `val.jsonl` (carpeta `backend/training/data/`).
Opción B: si el repositorio es público, descárgalos con la URL raw de GitHub.

In [ ]:
import os
REPO_RAW = ""  # p. ej. "https://raw.githubusercontent.com/<usuario>/senda-mars/<rama>/backend/training/data"

if REPO_RAW:
    !wget -q {REPO_RAW}/train.jsonl -O train.jsonl
    !wget -q {REPO_RAW}/val.jsonl -O val.jsonl
else:
    from google.colab import files
    print("Sube train.jsonl y val.jsonl")
    files.upload()

assert os.path.exists("train.jsonl") and os.path.exists("val.jsonl"), "Faltan los archivos de datos"
!wc -l train.jsonl val.jsonl

## 3. Cargar Qwen3 1.7B en 4 bits

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 4096   # prompt de sistema + herramientas + documentación + respuesta
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

## 4. Adaptadores LoRA

Solo se entrena ~1–2 % de los pesos: cabe en la T4 y conserva lo que el modelo ya sabe.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

## 5. Formatear EXACTAMENTE como Ollama

Cada turno del asistente se convierte en un ejemplo `prompt → completion`, donde el prompt es
**idéntico** al que Ollama arma en producción con `think: false`:

- a la última orden del usuario se le agrega ` /no_think`;
- el turno del asistente empieza con un bloque vacío `<think>

</think>

`;
- los turnos anteriores (llamadas a herramientas, resultados) van como historial, sin ese bloque.

En la ronda 1 el bloque vacío solo aparecía antes de la respuesta final, y el modelo aprendió
"bloque vacío → responder con texto": por eso casi no llamaba herramientas en Ollama.

In [ ]:
import json
from datasets import Dataset

def ollama_examples(sample):
    msgs = [dict(m) for m in sample["messages"]]
    tools = sample["tools"]
    last_user = max(i for i, m in enumerate(msgs) if m["role"] == "user")
    msgs[last_user]["content"] = msgs[last_user]["content"] + " /no_think"   # como Ollama
    out = []
    for j in range(last_user + 1, len(msgs)):
        m = msgs[j]
        if m["role"] != "assistant":
            continue
        prompt = tokenizer.apply_chat_template(msgs[:j], tools=tools, tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
        if m.get("tool_calls"):
            body = "\n".join("<tool_call>\n" + json.dumps({"name": c["function"]["name"],
                                                           "arguments": c["function"]["arguments"]},
                                                          ensure_ascii=False) + "\n</tool_call>"
                             for c in m["tool_calls"])
        else:
            body = m["content"]
        out.append({"prompt": prompt, "completion": body + "<|im_end|>"})
    return out

def build(path):
    rows = [json.loads(line) for line in open(path, encoding="utf-8")]
    return Dataset.from_list([ex for r in rows for ex in ollama_examples(r)])

train_ds, val_ds = build("train.jsonl"), build("val.jsonl")
n_tokens = lambda ex: len(tokenizer(ex["prompt"] + ex["completion"]).input_ids)
train_ds = train_ds.map(lambda ex: {"n_tokens": n_tokens(ex)})
val_ds = val_ds.map(lambda ex: {"n_tokens": n_tokens(ex)})
print("ejemplos:", len(train_ds), "train /", len(val_ds), "val · demasiado largos:",
      sum(1 for x in train_ds["n_tokens"] if x > MAX_SEQ))
train_ds = train_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ).remove_columns("n_tokens")
val_ds = val_ds.filter(lambda x: x["n_tokens"] <= MAX_SEQ).remove_columns("n_tokens")

ex = train_ds[0]
assert ex["prompt"].endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"), "el prompt no termina como en Ollama"
assert " /no_think<|im_end|>" in ex["prompt"]
print(ex["prompt"][-500:], "\n--- completion ---\n", ex["completion"])

## 6. Entrenar

Con ejemplos `prompt → completion`, la pérdida se calcula **solo sobre la completion** (lo que el
modelo debe generar: la llamada a la herramienta o la respuesta). El prompt solo se lee.

In [ ]:
from trl import SFTTrainer, SFTConfig

EPOCHS = 1   # ~2 ejemplos por conversación: con 1 época ya ve cada caso dos veces

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        completion_only_loss=True,
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=150,
        save_strategy="no",
        optim="adamw_8bit",
        seed=7,
        report_to="none",
        output_dir="outputs",
    ),
)
stats = trainer.train()
print(stats)

## 7. Probar antes de exportar

In [ ]:
import json
FastLanguageModel.for_inference(model)

sample = json.loads(open("val.jsonl", encoding="utf-8").readline())
system, tools = sample["messages"][0], sample["tools"]

def ask(question):
    msgs = [system, {"role": "user", "content": question + " /no_think"}]
    prompt = tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, temperature=0.2, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=False)

for q in ["Arma la misión de hoy, tenemos dos horas", "¿Cómo pongo una zona de peligro?",
          "How much oxygen is left on the walk?", "¿Cuál es la capital de Francia?"]:
    print("›", q, "\n", ask(q), "\n")

## 8. Exportar a GGUF (para Ollama)

Fusiona los adaptadores con el modelo y lo cuantiza a `q4_k_m` (~1.1 GB), el mismo formato del
modelo base. Al terminar, descarga el archivo `.gguf`.

In [ ]:
model.save_pretrained_gguf("senda-copilot", tokenizer, quantization_method="q4_k_m")
!ls -lh senda-copilot*/*.gguf senda-copilot*.gguf 2>/dev/null

In [ ]:
import glob
from google.colab import files
gguf = sorted(glob.glob("senda-copilot*/*.gguf") + glob.glob("senda-copilot*.gguf"))
print(gguf)
files.download([g for g in gguf if "q4_k_m" in g.lower()][0])

## 9. En tu PC

```powershell
cd backend
python training/install_in_ollama.py ruta\al\senda-copilot.Q4_K_M.gguf   # crea el modelo "senda-copilot"
python eval/run_eval.py --model senda-copilot                            # examen: compara con qwen3:1.7b
```

Si la nota mejora, en `backend/.env` pon `SENDA_LLM_MODEL=senda-copilot` y reinicia el backend.